# Analyse du trafic web

TP d'analyse de données — exploration, segmentation et cohortes.

Données : `data/processed/trafic_nettoye.csv` (2000 sessions, 7 colonnes).

## 1. Chargement et contrôle des données

On charge le CSV et on vérifie que tout est ok (dimensions, valeurs manquantes, doublons).

In [ ]:
# Installation des dépendances (à exécuter une seule fois)
# Depuis la racine du projet : pip install -r requirements.txt
# Ou dans le notebook :
%pip install -q -r ../requirements.txt

In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

In [ ]:
# chemin vers le dataset
DATA_PATH = Path("../data/processed/trafic_nettoye.csv")

# charge les données dans un dataframe
df = pd.read_csv(DATA_PATH)

df.head()

In [ ]:
# dimensions du dataframe
print(f"Dimensions : {df.shape}")
if df.shape != (2000, 7):
    print("Attention : shape différente de (2000, 7)")

# types et info
print("\nTypes :")
print(df.dtypes)
df.info()

# valeurs manquantes
nulls = df.isnull().sum()
print(f"\nValeurs manquantes :\n{nulls}")
if nulls.sum() > 0:
    print("Il y a des valeurs manquantes à traiter")

# doublons
n_doublons = df.duplicated().sum()
print(f"\nDoublons : {n_doublons}")

# sources de trafic
print(f"\nSources ({df['traffic_source'].nunique()}) : {sorted(df['traffic_source'].unique())}")

### Bilan

- 2000 lignes, 7 colonnes
- Pas de valeurs manquantes ni de doublons
- 6 variables numériques + `traffic_source` (Direct, Organic, Paid, Referral, Social)

**Note :** il n'y a pas de date dans les données. Pour la partie cohortes, on utilisera `previous_visits` (nombre de visites passées) à la place.

In [ ]:
# résumé statistique des colonnes numériques
df.describe()

## 2. Exploration globale des données

On regarde les distributions des variables, la répartition des sources de trafic et les corrélations.

In [ ]:
# histogrammes des variables numériques
cols_num = ["page_views", "session_duration", "bounce_rate", "time_on_page", "previous_visits", "conversion_rate"]
df[cols_num].hist(figsize=(12, 8))
plt.tight_layout()
plt.show()

In [ ]:
# répartition par source de trafic
df["traffic_source"].value_counts().plot(kind="bar", title="Répartition par source")
plt.ylabel("Nombre de sessions")
plt.show()

print(df["traffic_source"].value_counts())

In [ ]:
# matrice de corrélation
plt.figure(figsize=(8, 6))
sns.heatmap(df[cols_num].corr(), annot=True, cmap="Blues")
plt.title("Corrélations entre variables")
plt.show()

In [ ]:
# boxplots pour repérer les valeurs extrêmes
df[["page_views", "session_duration", "bounce_rate", "time_on_page"]].boxplot(figsize=(10, 5))
plt.title("Boxplots")
plt.show()

### Ce qu'on observe

- **Organic** est la source la plus fréquente.
- La durée de session varie beaucoup (certaines sessions très courtes).
- Le rebond moyen tourne autour de 0,3.
- Les boxplots montrent des valeurs extrêmes sur `session_duration`.

**Question ouverte :** pourquoi Organic domine-t-il autant ? Est-ce lié au SEO, au budget pub, ou au type de site ? À creuser avec d'autres données qu'on n'a pas ici.

## 3. Analyse par segments

### 3.1 Par source de trafic

On calcule la moyenne de chaque métrique par source de trafic.

In [ ]:
# moyennes par source
segments_source = df.groupby("traffic_source").agg({
    "page_views": "mean",
    "session_duration": "mean",
    "bounce_rate": "mean",
    "time_on_page": "mean",
    "previous_visits": "mean",
    "conversion_rate": "mean",
}).round(3)

segments_source

In [ ]:
# rebond par source
segments_source["bounce_rate"].plot(kind="bar", title="Taux de rebond moyen par source")
plt.ylabel("bounce_rate")
plt.show()

# conversion par source
segments_source["conversion_rate"].plot(kind="bar", title="Taux de conversion moyen par source", color="green")
plt.ylabel("conversion_rate")
plt.show()

# pages vues par source
segments_source["page_views"].plot(kind="bar", title="Pages vues moyennes par source", color="steelblue")
plt.ylabel("page_views")
plt.show()

### 3.2 Nouveau vs récurrent

- **Nouveau** : `previous_visits == 0`
- **Récurrent** : au moins 1 visite antérieure

In [ ]:
# création du segment
df["type_visiteur"] = "Récurrent"
df.loc[df["previous_visits"] == 0, "type_visiteur"] = "Nouveau"

df["type_visiteur"].value_counts()

In [ ]:
# comparaison Nouveau vs Récurrent
df.groupby("type_visiteur").agg({
    "page_views": "mean",
    "bounce_rate": "mean",
    "conversion_rate": "mean",
}).round(3)

In [ ]:
# croisement type visiteur × source
df.groupby(["type_visiteur", "traffic_source"]).agg({
    "page_views": "mean",
    "bounce_rate": "mean",
    "conversion_rate": "mean",
}).round(3)

### 3.3 Engagement et rebond

- **Engagement** : tertiles de `page_views` → Faible / Moyen / Fort
- **Rebond** : Bas (< 0,2) / Moyen (0,2–0,4) / Haut (≥ 0,4)

In [ ]:
# niveau d'engagement (3 groupes égaux)
df["niveau_engagement"] = pd.qcut(df["page_views"], q=3, labels=["Faible", "Moyen", "Fort"])

# niveau de rebond (seuils fixes)
df["niveau_rebond"] = "Moyen"
df.loc[df["bounce_rate"] < 0.2, "niveau_rebond"] = "Bas"
df.loc[df["bounce_rate"] >= 0.4, "niveau_rebond"] = "Haut"

print(df["niveau_engagement"].value_counts())
print()
print(df["niveau_rebond"].value_counts())

In [ ]:
# métriques par niveau d'engagement
df.groupby("niveau_engagement").agg({
    "page_views": "mean",
    "bounce_rate": "mean",
    "conversion_rate": "mean",
}).round(3)

In [ ]:
# métriques par niveau de rebond
df.groupby("niveau_rebond").agg({
    "page_views": "mean",
    "bounce_rate": "mean",
    "conversion_rate": "mean",
}).round(3)

### Ce qu'on observe (segments)

- **Referral** a le meilleur rebond et la meilleure conversion.
- **Paid** et **Social** rebondissent plus.
- Les visiteurs **Récurrents** convertissent mieux que les **Nouveaux**.
- Plus l'engagement est fort, plus la conversion monte.

## 4. Analyse par cohortes

Pas de date dans les données, donc on groupe par fidélité (`previous_visits`).

### 4.1 Cohortes de fidélité

- **Nouveau** : `previous_visits == 0`
- **Occasionnel** : 1 ou 2 visites passées
- **Régulier** : 3 ou 4
- **Fidèle** : 5 et plus

In [ ]:
# création de la colonne cohorte
df["cohorte"] = "Fidèle"
df.loc[df["previous_visits"] == 0, "cohorte"] = "Nouveau"
df.loc[df["previous_visits"].isin([1, 2]), "cohorte"] = "Occasionnel"
df.loc[df["previous_visits"].isin([3, 4]), "cohorte"] = "Régulier"

df["cohorte"].value_counts()

In [ ]:
# moyennes par cohorte
cohortes_fidelite = df.groupby("cohorte").agg({
    "page_views": "mean",
    "session_duration": "mean",
    "bounce_rate": "mean",
    "time_on_page": "mean",
    "previous_visits": "mean",
    "conversion_rate": "mean",
}).round(3)

cohortes_fidelite

In [ ]:
cohortes_fidelite["conversion_rate"].plot(kind="line", marker="o", title="Conversion par cohorte")
plt.ylabel("conversion_rate")
plt.show()

### 4.2 Cohorte × Source

On croise la cohorte et la source de trafic pour voir d'où viennent les nouveaux visiteurs et comment évolue la conversion.

In [ ]:
# croisement cohorte × source
matrice_cohorte_source = df.groupby(["cohorte", "traffic_source"]).agg({
    "page_views": "mean",
    "session_duration": "mean",
    "bounce_rate": "mean",
    "time_on_page": "mean",
    "previous_visits": "mean",
    "conversion_rate": "mean",
}).round(3)

matrice_cohorte_source

In [ ]:
# nouveaux visiteurs par source
df[df["cohorte"] == "Nouveau"]["traffic_source"].value_counts().plot(
    kind="bar", title="Nouveaux visiteurs par source", color="coral"
)
plt.ylabel("nb_sessions")
plt.show()

### Ce qu'on observe (cohortes)

- La conversion augmente quand le visiteur revient plus souvent.
- **Organic** amène le plus de nouveaux visiteurs.
- **Referral** reste performant à tous les niveaux de fidélité.

**À creuser :** est-ce que Paid vaut le coup malgré un rebond plus élevé ? Il faudrait le coût par clic pour trancher.

## 5. Synthèse et export

Récap de ce qu'on a trouvé + export des tableaux pour Tableau / Power BI.

### Principaux résultats

1. **Organic** est le canal principal (~40 % des sessions).
2. **Referral** performe le mieux (rebond bas, conversion haute).
3. **Paid** et **Social** ont un rebond plus élevé — à surveiller.
4. Les **nouveaux visiteurs** convertissent moins bien que les récurrents.
5. Plus un visiteur revient, plus il convertit (cohortes de fidélité).

Pour la viz dans Tableau / Power BI, je prévois :
- un bar chart rebond/conversion par source ;
- une courbe conversion par cohorte ;
- une heatmap cohorte × source.

In [ ]:
# export des 3 tableaux pour la suite (Tableau / Power BI)
segments_source.reset_index().to_csv("../data/processed/segments_par_source.csv", index=False)
cohortes_fidelite.reset_index().to_csv("../data/processed/cohortes_par_fidelite.csv", index=False)
matrice_cohorte_source.reset_index().to_csv("../data/processed/matrice_cohorte_source.csv", index=False)
print("Fichiers exportés dans data/processed/")